# Step 8: Train the Model

**SageMaker Unified Studio Component**: Training Jobs

**What you'll learn**: Train and evaluate a logistic regression model

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')

## Load Feature Dataset

In [ ]:
s3_path = f's3://{bucket_name}/data/features/features.parquet'
df = pd.read_parquet(s3_path)
print(f"Loaded {len(df):,} samples")
df.head()

## Train/Test Split

In [ ]:
X = df[['temperature', 'temp_diff']]
y = df['overheat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"\nClass distribution (train):")
print(y_train.value_counts(normalize=True))

## Train Logistic Regression

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("✓ Model trained successfully!")

## Evaluate the Model

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Overheat']))

In [ ]:
# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("\n[[TN  FP]")
print(" [FN  TP]]")

## Inspect Model Coefficients

In [ ]:
coef_df = pd.DataFrame({
    'feature': ['temperature', 'temp_diff'],
    'coefficient': model.coef_[0]
})
print("Model Coefficients:")
print(coef_df)
print(f"\nIntercept: {model.intercept_[0]:.3f}")

## Save the Model

In [ ]:
import tarfile
import boto3

# Save locally
model_path = 'model.pkl'
joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")

# Create tar.gz for SageMaker (required format for model registry)
tar_path = 'model.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(model_path)
print(f"Created: {tar_path}")

# Upload both to S3
s3 = boto3.client('s3')

# Upload pkl
s3_pkl_path = 'models/logistic_regression/model.pkl'
s3.upload_file(model_path, bucket_name, s3_pkl_path)
print(f"Uploaded to: s3://{bucket_name}/{s3_pkl_path}")

# Upload tar.gz (required for SageMaker endpoints)
s3_tar_path = 'models/logistic_regression/model.tar.gz'
s3.upload_file(tar_path, bucket_name, s3_tar_path)
print(f"Uploaded to: s3://{bucket_name}/{s3_tar_path}")

## Save Test Data for Validation

In [ ]:
# Save test set for later validation
X_test.to_parquet(f's3://{bucket_name}/data/features/test_features.parquet', index=False)
y_test.to_frame().to_parquet(f's3://{bucket_name}/data/features/test_labels.parquet', index=False)
print("Test data saved for validation")

## Key Insights

**Model performance**: ~90-95% accuracy

**Feature importance**: Both temperature and temp_diff contribute

**Next step**: Track this experiment with MLflow